In [1]:

#################################################################################################################################################################################################################################################################################
#CONEXION AL ORACLE

import oracledb
import pandas as pd
from sqlalchemy import create_engine
import sys
from sqlalchemy import create_engine
from sqlalchemy import text
import sys
from decouple import config

oracledb.init_oracle_client()
oracledb.version = "8.3.0"
sys.modules["cx_Oracle"] = oracledb
un = config("USER4_BDI_POSTGRES")
pw = config("PASS4_BDI_POSTGRES")
hostname=config("HOST4_BDI_POSTGRES")
service_name="WNET"
port = 1527

engine2 = create_engine(f'oracle://{un}:{pw}@',connect_args={
        "host": hostname,
        "port": port,
        "service_name": service_name
    }
)

connection2 = engine2.connect()

base2 = pd.read_sql_query(f"SELECT * FROM cmdia10", con=connection2)

connection2.close()




In [2]:
base2

,diagcod,diagdes,diagraiz,diagniv,diagsexcob,indplancapcobcod,diagenftra,diagregtipo,diagacptflg,estregcod,edxcapcod,edxgpocod,edxsgpcod,diaggpcflg,diagenfrara,diagescglaflg,diagpeasflg,diagdengflg
0,A63.0,VERRUGAS (VENEREAS) ANOGENITALES,01,0500,2,3,0,1,1,1,01,05,00,NaN,NaN,NaN,NaN,NaN
1,A63.8,OTRAS ENFERMEDADES DE TRANSMISION PREDOMINANTE...,01,0500,2,3,0,1,1,1,01,05,00,NaN,NaN,NaN,NaN,NaN
2,A64,ENFERMEDAD DE TRANSMISION SEXUAL NO ESPECIFICADA,01,0500,2,3,0,0,1,1,01,05,00,NaN,NaN,NaN,NaN,NaN
3,A65,SIFILIS NO VENEREA,01,0600,2,3,0,0,1,1,01,06,00,NaN,NaN,NaN,NaN,NaN
4,A66,FRAMBESIA,01,0600,2,3,0,0,0,1,01,06,00,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14319,G03.0,MENINGITIS APIOGENA,06,0100,2,3,0,1,1,1,06,01,00,NaN,0.0,1.0,NaN,NaN
14320,I67.8,OTRAS ENFERMEDADES CEREBROVASCULARES ESPECIFIC...,09,0700,2,3,0,1,1,1,09,07,00,NaN,0.0,1.0,NaN,NaN
14321,D33.0,"TUMOR BENIGNO DEL ENCEFALO, SUPRATENTORIAL",02,0300,2,3,0,1,1,1,02,03,00,NaN,1.0,1.0,NaN,NaN
14322,Q93.6,SUPRESIONES VISIBLES SOLO EN LA PROMETAFASE,17,1100,2,3,0,1,1,1,17,11,00,NaN,1.0,NaN,NaN,NaN


In [3]:
base2.columns

Index(['diagcod', 'diagdes', 'diagraiz', 'diagniv', 'diagsexcob',
       'indplancapcobcod', 'diagenftra', 'diagregtipo', 'diagacptflg',
       'estregcod', 'edxcapcod', 'edxgpocod', 'edxsgpcod', 'diaggpcflg',
       'diagenfrara', 'diagescglaflg', 'diagpeasflg', 'diagdengflg'],
      dtype='object')

In [4]:

#################################################################################################################################################################################################################################################################################
#CREAMOS LA TABLA TEMPORAL Y LA SUBIMOS AL POSTGRES SQL

DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="dl_essi"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena3  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine3 = create_engine(cadena3)
connection3 = engine3.connect()



#connection3.execute('CREATE TEMPORARY TABLE tmp_cmdia10 ()')
base2.to_sql(name='tmp_cmdia10', con=engine3, if_exists='replace', index=False)



#################################################################################################################################################################################################################################################################################
#ACTUALIZAMOS EL cmaho10 FALSO CON LO OBTENIDO DEL ORACLE


query="""
UPDATE cmdia10 
SET 
diagdes =t.diagdes,
diagraiz =t.diagraiz,
diagniv =t.diagniv,
diagsexcob =t.diagsexcob,
indplancapcobcod =t.indplancapcobcod,
diagenftra =t.diagenftra,
diagregtipo =t.diagregtipo,
diagacptflg =t.diagacptflg,
estregcod =t.estregcod,
edxcapcod =t.edxcapcod,
edxgpocod =t.edxgpocod,
edxsgpcod =t.edxsgpcod,
diaggpcflg =t.diaggpcflg,
diagenfrara =t.diagenfrara

FROM tmp_cmdia10 t 
WHERE cmdia10.diagcod = t.diagcod;

INSERT INTO cmdia10 (diagcod,diagdes,diagraiz,diagniv,diagsexcob,indplancapcobcod,diagenftra,diagregtipo,diagacptflg,estregcod,edxcapcod,edxgpocod,edxsgpcod,diaggpcflg,diagenfrara) 

SELECT diagcod,diagdes,diagraiz,diagniv,diagsexcob,indplancapcobcod,diagenftra,diagregtipo,diagacptflg,estregcod,edxcapcod,edxgpocod,edxsgpcod,diaggpcflg,diagenfrara

FROM tmp_cmdia10 
WHERE NOT EXISTS (
    SELECT 1 
    FROM cmdia10 
    WHERE cmdia10.diagcod = tmp_cmdia10.diagcod
);


"""

c1= text(query)
connection3.execute(c1)




#BORRAMOS LAS TABLAS
base2 = pd.read_sql_query(f"SELECT * FROM cmdia10", con=connection3)
query2="""
DROP TABLE tmp_cmdia10;
"""

c2= text(query2)
connection3.execute(c2)
connection3.close()


In [5]:

#################################################################################################################################################################################################################################################################################

#SUBIMOS ESA TABLA ACTUALIZADA COMO TEMPORAL A LA BASE DEL DIM ACTIVIT

DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="dw_essalud"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena4  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

engine4 = create_engine(cadena4)
connection4 = engine4.connect()


base2.to_sql(name='tmp_cmdia10', con=engine4, if_exists='replace', index=False)



#################################################################################################################################################################################################################################################################################
#ACTUALIZAMOS EL DIM ACTIVITI FALSO CON LA BASE DE DATOS QUE SUBIMOS

query="""
UPDATE dim_cie10 
SET des_cie      = t.diagdes

FROM tmp_cmdia10 t 
WHERE dim_cie10.cod_cie = t.diagcod;

INSERT INTO dim_cie10 (cod_cie, des_cie) 
SELECT diagcod,diagdes
FROM tmp_cmdia10 
WHERE NOT EXISTS (
    SELECT 1 
    FROM dim_cie10 
    WHERE dim_cie10.cod_cie = tmp_cmdia10.diagcod
);

DROP TABLE tmp_cmdia10;
"""
c1= text(query)
connection4.execute(c1)
connection4.close()


IntegrityError: (psycopg2.errors.UniqueViolation) llave duplicada viola restricción de unicidad «dim_cie10_pkey»
DETAIL:  Ya existe la llave (id_cie)=(14299).

[SQL: 
UPDATE dim_cie10 
SET des_cie      = t.diagdes

FROM tmp_cmdia10 t 
WHERE dim_cie10.cod_cie = t.diagcod;

INSERT INTO dim_cie10 (cod_cie, des_cie) 
SELECT diagcod,diagdes
FROM tmp_cmdia10 
WHERE NOT EXISTS (
    SELECT 1 
    FROM dim_cie10 
    WHERE dim_cie10.cod_cie = tmp_cmdia10.diagcod
);

DROP TABLE tmp_cmdia10;
]
(Background on this error at: https://sqlalche.me/e/14/gkpj)